In [3]:
"""COVID vs non-COVID classification: PCA + Logistic Regression / Random Forest / linear SVM
Run A = all genes (as requested).  Run B = robustness check on reliably-expressed genes only."""
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

SEED, OUT = 42, "Output.csv"
df = pd.read_csv("Final_Dataset_AIBD.csv", low_memory=False)

# Drop rows that are completely empty or contain NaNs to prevent PCA/model crashes
df = df.dropna(subset=df.columns.difference(["Age", "Gender", "ICU"]))

# Label from sample ID: C* = COVID (1), NC* = non-COVID (0)
ids = df["Unnamed: 0"].astype(str)
y = (~ids.str.startswith("NC")).astype(int).values

# Features = gene expression only. ICU excluded (NaN for every non-COVID sample = label leak);
# Age/Gender excluded (COVID group is older -> confounder).
X_all = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0", "Age", "Gender", "ICU"]).astype(float)
X_all = X_all.loc[:, X_all.var() > 0]

def slog(a):  # signed log: tames heavy tails and handles the negative values in ~2,200 genes
    return np.sign(a) * np.log1p(np.abs(a))

def make(m):  # normalisation INSIDE the pipeline -> fitted on training folds only (no leakage)
    return Pipeline([("log", FunctionTransformer(slog)), ("scale", StandardScaler()), ("clf", m)])

colors = {"Logistic Regression": "#e63946", "Random Forest": "#2a9d8f", "SVM (linear)": "#457b9d"}
def get_models():
    return {"Logistic Regression": LogisticRegression(C=0.01, max_iter=5000, random_state=SEED),
            "Random Forest": RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1),
            "SVM (linear)": SVC(kernel="linear", C=0.001, random_state=SEED)}
def score(pipe, X):
    return pipe.predict_proba(X)[:, 1] if hasattr(pipe, "predict_proba") else pipe.decision_function(X)

def run(X, tag):
    genes = X.columns.to_numpy(); Xv = X.values
    print(f"\n===== RUN {tag}: {Xv.shape[0]} samples x {Xv.shape[1]} genes =====")

    # ---------- PCA on the whole data ----------
    Xn = StandardScaler().fit_transform(slog(Xv))
    pca = PCA(n_components=20, random_state=SEED).fit(Xn); Z = pca.transform(Xn); evr = pca.explained_variance_ratio_
    print("PCA var% PC1-5:", np.round(evr[:5]*100, 1), "| PC1-10 cum: %.1f%%" % (evr[:10].sum()*100))
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
    for lab, nm, c in [(0, "Non-COVID", "#2a9d8f"), (1, "COVID", "#e63946")]:
        ax[0].scatter(Z[y==lab, 0], Z[y==lab, 1], s=28, alpha=.75, c=c, label=nm, edgecolor="white", lw=.3)
        ax[1].scatter(Z[y==lab, 1], Z[y==lab, 2], s=28, alpha=.75, c=c, label=nm, edgecolor="white", lw=.3)
    ax[0].set(xlabel=f"PC1 ({evr[0]*100:.1f}%)", ylabel=f"PC2 ({evr[1]*100:.1f}%)", title=f"PCA PC1 vs PC2 [{tag}]"); ax[0].legend()
    ax[1].set(xlabel=f"PC2 ({evr[1]*100:.1f}%)", ylabel=f"PC3 ({evr[2]*100:.1f}%)", title="PCA PC2 vs PC3")
    ax[2].bar(range(1, 21), evr*100, color="#457b9d", label="Per PC")
    ax[2].plot(range(1, 21), np.cumsum(evr)*100, "o-", c="#e63946", ms=3, label="Cumulative")
    ax[2].set(xlabel="Principal component", ylabel="Explained variance (%)", title="Scree plot"); ax[2].legend()
    plt.tight_layout(); plt.savefig(OUT+f"{tag}_1_pca.png", dpi=150); plt.close()
    load = pd.DataFrame(pca.components_[:3].T, index=genes, columns=["PC1", "PC2", "PC3"])
    load.reindex(load["PC1"].abs().sort_values(ascending=False).index).head(25).to_csv(OUT+f"{tag}_pca_top_loadings_PC1.csv")
    pd.DataFrame({"PC": range(1, 21), "explained_var_pct": evr*100}).to_csv(OUT+f"{tag}_pca_variance.csv", index=False)

    # ---------- Models: 5-fold CV (all data) + 80/20 hold-out ----------
    Xtr, Xte, ytr, yte = train_test_split(Xv, y, test_size=0.2, stratify=y, random_state=SEED)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.5)); rows, imps = [], {}
    for name, m in get_models().items():
        meth = "predict_proba" if hasattr(m, "predict_proba") else "decision_function"
        s_cv = cross_val_predict(make(m), Xv, y, cv=cv, method=meth); s_cv = s_cv[:, 1] if s_cv.ndim > 1 else s_cv
        auc_cv = roc_auc_score(y, s_cv); f, t, _ = roc_curve(y, s_cv)
        ax[0].plot(f, t, c=colors[name], lw=2, label=f"{name} (AUC = {auc_cv:.3f})")
        pipe = make(get_models()[name]).fit(Xtr, ytr); s_te = score(pipe, Xte)
        auc_te = roc_auc_score(yte, s_te); f, t, _ = roc_curve(yte, s_te)
        ax[1].plot(f, t, c=colors[name], lw=2, label=f"{name} (AUC = {auc_te:.3f})")
        rows.append(dict(Model=name, CV_AUC_5fold=round(auc_cv, 4), Holdout_AUC=round(auc_te, 4),
                         Holdout_Accuracy=round(accuracy_score(yte, pipe.predict(Xte)), 4)))
        print(rows[-1])
        clf = make(get_models()[name]).fit(Xv, y).named_steps["clf"]   # importances: fit on all data
        imps[name] = pd.Series(clf.feature_importances_ if hasattr(clf, "feature_importances_") else clf.coef_.ravel(), index=genes)
    for a, t in zip(ax, [f"ROC - 5-fold CV, all {len(y)} samples [{tag}]", f"ROC - 20% hold-out (n={len(yte)})"]):
        a.plot([0, 1], [0, 1], "k--", lw=1); a.set(xlabel="False positive rate", ylabel="True positive rate", title=t); a.legend(loc="lower right")
    plt.tight_layout(); plt.savefig(OUT+f"{tag}_2_roc_auc.png", dpi=150); plt.close()
    pd.DataFrame(rows).to_csv(OUT+f"{tag}_model_performance.csv", index=False)

    # ---------- Feature importance ----------
    fig, ax = plt.subplots(1, 3, figsize=(19, 7))
    for a, (name, s) in zip(ax, imps.items()):
        top = s.reindex(s.abs().sort_values(ascending=False).index).head(20)[::-1]
        a.barh(top.index, top.values, color=[colors[name] if v >= 0 else "#999999" for v in top.values])
        a.set_title(f"{name}\n" + ("(Gini importance)" if "Forest" in name else "(coefficient: + = COVID, - = non-COVID)"))
        a.tick_params(axis="y", labelsize=8)
    plt.tight_layout(); plt.savefig(OUT+f"{tag}_3_feature_importance.png", dpi=150); plt.close()
    imp = pd.DataFrame(imps); imp.to_csv(OUT+f"{tag}_feature_importance_all_genes.csv")
    rk = pd.DataFrame({n+"_rank": imp[n].abs().rank(ascending=False, method="min").astype(int) for n in imp})
    cons = pd.concat([imp, rk], axis=1); cons["mean_rank"] = rk.mean(axis=1)
    cons.sort_values("mean_rank").head(30).to_csv(OUT+f"{tag}_top30_consensus_genes.csv")
    for n in imp: print(n, "top10:", imp[n].abs().sort_values(ascending=False).head(10).index.tolist())
    print("consensus top10:", cons.sort_values("mean_rank").head(10).index.tolist())

# ---------------- Run A: all genes ----------------
run(X_all, "A_all_genes")

# ---------------- Run B: reliably-expressed genes only ----------------
# drop genes detected (>0.1) in <25% of samples (near-zero genes carry the batch fingerprint)
# and 5S rRNA genes (technical, extremely high-variance)
keep = (X_all > 0.1).mean() >= 0.25
keep &= ~X_all.columns.str.match(r"^(RNA5S|RNA18S|RNA28S|RNA45S)")
run(X_all.loc[:, keep], "B_filtered_genes")


===== RUN A_all_genes: 185 samples x 17721 genes =====
PCA var% PC1-5: [28.5 15.6  3.6  2.8  2.3] | PC1-10 cum: 59.8%
{'Model': 'Logistic Regression', 'CV_AUC_5fold': np.float64(0.9996), 'Holdout_AUC': np.float64(1.0), 'Holdout_Accuracy': 0.9189}
{'Model': 'Random Forest', 'CV_AUC_5fold': np.float64(0.99), 'Holdout_AUC': np.float64(0.9941), 'Holdout_Accuracy': 0.9459}
{'Model': 'SVM (linear)', 'CV_AUC_5fold': np.float64(0.9994), 'Holdout_AUC': np.float64(1.0), 'Holdout_Accuracy': 0.9189}
Logistic Regression top10: ['AQP5', 'LCE3C', 'XAGE1A', 'RNA5S9', 'RNA5S5', 'RNA5S2', 'RNA5S10', 'RNA5S7', 'RNA5S14', 'RNA5S8']
Random Forest top10: ['MCM10', 'SHCBP1', 'DTL', 'ESCO2', 'HASPIN', 'PALM2AKAP2', 'RNA5S8', 'RNA5S2', 'SKA1', 'MKI67']
SVM (linear) top10: ['AQP5', 'LCE3C', 'XAGE1A', 'C2CD4A', 'CACNG7', 'URGCP-MRPS24', 'RNA5S7', 'RNA5S4', 'RNA5S5', 'RNA5S8']
consensus top10: ['RNA5S8', 'RNA5S2', 'RNA5S7', 'RNA5S12', 'RNA5S1', 'RNA5S10', 'RNA5S16', 'RNA5S15', 'RNA5S14', 'RNA5S5']

===== RUN B_f

In [4]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, roc_auc_score

SEED = 42
df = pd.read_csv("Final_Dataset_AIBD.csv", low_memory=False)
y = (~df["Unnamed: 0"].astype(str).str.startswith("NC")).astype(int).values
X = df.drop(columns=["Unnamed: 0.1","Unnamed: 0","Age","Gender","ICU"]).astype(float)
X = X.loc[:, X.var() > 0].values
slog = lambda a: np.sign(a) * np.log1p(np.abs(a))

def models():
    return {"Logistic Regression": LogisticRegression(C=0.01, max_iter=5000, random_state=SEED),
            "SVM (linear)": SVC(kernel="linear", C=0.001, random_state=SEED),
            "Random Forest": RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)}

def pipe(m, k):
    steps = [("log", FunctionTransformer(slog)), ("scale", StandardScaler())]
    if k: steps.append(("select", SelectKBest(f_classif, k=k)))   # selected on training data only
    return Pipeline(steps + [("clf", m)])

def metrics(label, yt, yp, s):
    tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
    P = tp/(tp+fp) if tp+fp else 0; R = tp/(tp+fn); S = tn/(tn+fp)
    return {"Model": label, "Accuracy": (tp+tn)/len(yt), "Precision": P, "Recall (Sensitivity)": R,
            "Specificity": S, "F1 score": 2*P*R/(P+R) if P+R else 0, "ROC-AUC": roc_auc_score(yt, s),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

def sc(p, Xt): return p.predict_proba(Xt)[:, 1] if hasattr(p, "predict_proba") else p.decision_function(Xt)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
res = {}
for setting, k in [("All genes", None), ("Top 25 genes", 25)]:
    h, c = [], []
    for name, m in models().items():
        lab = f"{name} ({setting.lower()})"
        p = pipe(m, k).fit(Xtr, ytr); h.append(metrics(lab, yte, p.predict(Xte), sc(p, Xte)))
        meth = "predict_proba" if hasattr(m, "predict_proba") else "decision_function"
        s = cross_val_predict(pipe(models()[name], k), X, y, cv=cv, method=meth)
        s = s[:, 1] if s.ndim > 1 else s
        thr = 0.5 if meth == "predict_proba" else 0.0
        c.append(metrics(lab, y, (s > thr).astype(int), s))
    res[f"holdout_{k or 'all'}"] = pd.DataFrame(h); res[f"cv_{k or 'all'}"] = pd.DataFrame(c)

pd.set_option("display.width", 220)
out = []
for key, title in [("holdout_all","HOLD-OUT (20%, n=37) - all genes"),("holdout_25","HOLD-OUT (20%, n=37) - top 25 genes"),
                   ("cv_all","5-FOLD CV (pooled, n=185) - all genes"),("cv_25","5-FOLD CV (pooled, n=185) - top 25 genes")]:
    t = res[key].round(4); print("\n"+title); print(t.to_string(index=False)); t.insert(0, "Setting", title); out.append(t)
pd.concat(out).to_csv("classification_metrics_table.csv", index=False)


HOLD-OUT (20%, n=37) - all genes
                          Model  Accuracy  Precision  Recall (Sensitivity)  Specificity  F1 score  ROC-AUC  TN  FP  FN  TP
Logistic Regression (all genes)    0.9189        1.0                  0.85          1.0    0.9189   1.0000  17   0   3  17
       SVM (linear) (all genes)    0.9189        1.0                  0.85          1.0    0.9189   1.0000  17   0   3  17
      Random Forest (all genes)    0.9459        1.0                  0.90          1.0    0.9474   0.9941  17   0   2  18

HOLD-OUT (20%, n=37) - top 25 genes
                             Model  Accuracy  Precision  Recall (Sensitivity)  Specificity  F1 score  ROC-AUC  TN  FP  FN  TP
Logistic Regression (top 25 genes)    0.9730        1.0                  0.95          1.0    0.9744   0.9971  17   0   1  19
       SVM (linear) (top 25 genes)    0.9459        1.0                  0.90          1.0    0.9474   0.9971  17   0   2  18
      Random Forest (top 25 genes)    0.9459        1.0    